### CWODMR Notebook

##### In this experiment we will use the Pulseblaster, AOM, RF Switch and the Windfreak
##### The pulseblaster outputs should be connected as listed below:
###### 1. Output 0: Connect to AOM
###### 2. Output 1: Connect to Lock-in AMP REF IN input
###### 3. Output 23: Connect to the RF Switch TTL input

In [1]:
from Ametek_5210 import Ametek5210
from pulseblaster_d import pulseblasterd
from windfreak import windfreak

In [2]:
import numpy as np
import time
import datetime
import matplotlib.pyplot as plt
import csv

In [3]:
#initalize equiptment
pulser = pulseblasterd('pb')
rf = windfreak('windfreak', 'COM5')
lockin = Ametek5210('lockin', 'GPIB0::12::INSTR')

In [4]:
# This function determines the delay based on the tc of the lock in amp
def set_delay(lockin_tc):
    timeconst = {0: '1 ms',
    1: '3 ms',
    2: '10 ms',
    3: '30 ms' ,
    4: '100 ms',
    5: '300 ms',
    6: '1 s',
    7: '3 s',
    8: '10 s',
    9: '30 s',
    10: '100 s',
    11: '300 s',
    12: '1 ks',
    13: '3 ks'}

    timeconst_bw = {'1 ms' :0,
    '3 ms': 1,
    '10 ms': 2,
    '30 ms': 3 ,
    '100 ms': 4,
    '300 ms' : 5,
    '1 s' : 6,
    '3 s' : 7,
    '10 s': 8,
    '30 s': 9,
    '100 s': 10,
    '300 s': 11,
    '1 ks' : 12,
    '3 ks' : 13}

    tc_s = {0: 0.001,  #get timeconstant in secs
    1: 0.003,
    2: 0.01,
    3: 0.03 ,
    4: 0.1,
    5: 0.3,
    6: 1,
    7: 3,
    8: 10,
    9: 30,
    10: 100,
    11: 300,
    12: 1000,
    13: 3000}

    val = timeconst_bw[lockin_tc]
    if(type(val) != int):
        return "Error: invalid input!"
    else:
        return 5* tc_s[val]
    

#### The CWODMR experiemnt is setup below:
#### 1. Make sure all the equipment is turned on
#### 2. There are some parameters in the beginning you can set to see how the results change based on the values you set

##### If you run into any issues please let the TA know!

In [ ]:
#TODO: set below paramters
rf_power = #in dBm, do not exceed -10dBm (try -40 to -10 dBm)
rf_step = 2 #MHz
rf_start_freq =  #feel free to adjust the range to see all the peaks (at 0 magnetic field, center of peaks are at 2870 MHz)
rf_stop_freq =   #MHz 

#setup pulser:
pulser._set_lock_in_width(2.5e-3)
pulser.program_CWstate()
pulser.start()

rf_freqs = np.arange(rf_start_freq, rf_stop_freq, rf_step)
#setup windfreak
rf.set_channel(0)
rf.set_power_dBm(rf_power)
#turn on windfreak
rf.on()

#create array of outputs from the lockin amp
vals = np.zeros(len(rf_freqs))
#set your time constant. At each point you will wait 5x this time constant
tc = '300 ms'

#set the sensitivity and the time constant of the lock in amp
lockin.sensitivity('30 mV') #30mV - you can use the smallest value that does not cause an overload error
time.sleep(1)
lockin.time_constant(tc) 
#it is a good idea to confirm that these were set visually by looking at the amplifier display

#set the delay based on the lockin time constant
delay = set_delay(tc)

for i in range(len(rf_freqs)):
    rf.set_freq(rf_freqs[i])
    time.sleep(int(delay)) #set using the time constant of the lockin amp later (5*timeconstant)
    val = lockin.magnitude()
    vals[i] = val
    print(val)

pulser.stop() #turn off the output signals
rf.off()

plt.plot(rf_freqs, vals)
plt.show()

In [ ]:
#write data to a csv file to save the data

dt = str(datetime.datetime.now()).strip().replace(' ', '-').replace('.','-').replace(':',"-")
filename = "cwodmr" + dt + ".csv"
header = ['rf_frequency(MHz)', 'Lock in amp mag']
with open(filename, 'w',newline='') as csvfile:
    csvwriter = csv.writer(csvfile)
    csvwriter.writerow(header)
    for i in range(len(vals)):
        row = [rf_freqs[i], vals[i]]
        csvwriter.writerow(row)
